# Silver: clean, validate and merge

**Layer:** Silver (ETL style: transform and validate first, then load only trusted rows)
**Input:** the bronze table for one source. **Output:** `silver.<source_name>` (trusted, one row per key) and `silver.<source_name>_rejects` (rows that failed a rule, with the reason).

## What this notebook does, in order
1. Read the source's entry from the config (same file as bronze)
2. Read bronze
3. Keep the **latest version** of each record (one row per merge key)
4. Apply type casts and standard cleaning
5. Check the validation rules. Rows that fail go to the rejects table, never silently dropped
6. Assert the key is unique
7. MERGE the good rows into silver (an older version never overwrites a newer one)
8. Write one row to `meta.run_log`, and re-raise on failure

## Rules this notebook follows
- Silver can be rebuilt from bronze at any time, without calling the API again.
- Rerunning is safe: the merge makes the result the same no matter how many times it runs.
- The watermark belongs to bronze (it tracks what was pulled from the API). Silver never touches it.

## 1. Parameters

In [ ]:
source_name = "github_issues"
config_path = "/lakehouse/default/Files/config/api_sources.json"

## 2. Imports

In [ ]:
import json
import uuid
import datetime as dt

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

## 3. Cleaning and validation rules (the part that differs per source)
These two dictionaries are the only source-specific code in the notebook. Adding a source means adding an entry here, not new logic.

- `CASTS`: column name to a Spark SQL expression that produces the clean value.
- `RULES`: a list of `(rule name, SQL condition that must be TRUE for a good row)`. A row that fails any rule is rejected with the names of the rules it failed.

Rules run **after** the casts, so they see the clean types. Conditions that evaluate to NULL count as a failure.

In [ ]:
CASTS = {
    "github_issues": {
        "number":     "CAST(number AS BIGINT)",
        "title":      "TRIM(title)",
        "state":      "LOWER(TRIM(state))",
        "created_at": "CAST(created_at AS TIMESTAMP)",
        "updated_at": "CAST(updated_at AS TIMESTAMP)",
        "user_login": "TRIM(user_login)",
    },
    "usgs_earthquakes": {
        "properties_mag":     "CAST(properties_mag AS DOUBLE)",
        "properties_place":   "TRIM(properties_place)",
        # USGS sends times as epoch milliseconds
        "properties_time":    "CAST(properties_time / 1000 AS TIMESTAMP)",
        "properties_updated": "CAST(properties_updated / 1000 AS TIMESTAMP)",
    },
}

RULES = {
    "github_issues": [
        ("id_not_null",        "id IS NOT NULL"),
        ("state_known",        "state IN ('open', 'closed')"),
        ("created_not_null",   "created_at IS NOT NULL"),
        ("updated_after_created", "updated_at >= created_at"),
    ],
    "usgs_earthquakes": [
        ("id_not_null",        "id IS NOT NULL"),
        ("updated_not_null",   "properties_updated IS NOT NULL"),
        ("magnitude_in_range", "properties_mag IS NULL OR properties_mag BETWEEN -2 AND 10"),
    ],
}

## 4. Helpers

In [ ]:
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)


def load_source(path, name):
    with open(path) as f:
        sources = json.load(f)
    src = next((s for s in sources if s.get("source_name") == name), None)
    if src is None:
        raise ValueError(f"source_name '{name}' not found in {path}")
    if name not in CASTS or name not in RULES:
        raise ValueError(f"Add '{name}' to CASTS and RULES in the rules cell before running silver.")
    return src


def log_run(**f):
    row = (f["run_id"], f["source_name"], "silver", None, f["status"],
           f.get("rows_read"), f.get("rows_written"), None, None, None,
           f.get("error_message"), f["started_at"], utc_now())
    schema = ("run_id string, source_name string, layer string, batch_id string, status string, "
              "rows_read bigint, rows_written bigint, pages_fetched int, watermark_start string, "
              "watermark_end string, error_message string, started_at timestamp, finished_at timestamp")
    spark.createDataFrame([row], schema).write.format("delta").mode("append").saveAsTable("meta.run_log")


def latest_per_key(df, key, order_col):
    """Keep one row per key: the newest version. Ties are broken by the most recent load."""
    w = Window.partitionBy(key).orderBy(F.col(order_col).desc(), F.col("ingest_ts").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


def apply_casts(df, casts):
    for column, expression in casts.items():
        df = df.withColumn(column, F.expr(expression))
    return df


def split_valid_rejected(df, rules):
    """Return (valid_rows, rejected_rows). A rejected row carries the names of every rule it failed."""
    failed = [F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(name)) for name, cond in rules]
    checked = df.withColumn("reject_reason", F.concat_ws("; ", *failed))   # concat_ws skips NULLs
    valid = checked.filter(F.col("reject_reason") == "").drop("reject_reason")
    rejected = (checked.filter(F.col("reject_reason") != "")
                       .withColumn("rejected_at", F.current_timestamp()))
    return valid, rejected

## 5. Main run

In [ ]:
def run(source_name):
    run_id, started_at = str(uuid.uuid4()), utc_now()
    info = dict(run_id=run_id, source_name=source_name, started_at=started_at)
    try:
        src = load_source(config_path, source_name)
        if not src.get("is_active", 1):
            print(f"{source_name} is switched off in the config. Nothing to do.")
            return

        key = src["merge_key"]
        order_col = src["watermark_field"].replace(".", "_")     # 'updated_at' or 'properties_updated'
        silver_table = f"silver.{source_name}"
        rejects_table = f"silver.{source_name}_rejects"

        # 1) Extract: read bronze, keep the newest version of each record
        bronze = spark.table(src["bronze_table"])
        rows_bronze = bronze.count()
        latest = latest_per_key(bronze, key, order_col)

        # 2) Transform: casts, then validation. Rules see the clean types.
        clean = apply_casts(latest, CASTS[source_name])
        valid, rejected = split_valid_rejected(clean, RULES[source_name])
        valid = valid.cache()
        n_valid, n_rejected = valid.count(), rejected.count()

        # Uniqueness is asserted, not assumed. Fail the run before writing anything.
        n_distinct = valid.select(key).distinct().count()
        if n_valid != n_distinct:
            raise RuntimeError(f"Key '{key}' is not unique after dedup: {n_valid} rows, {n_distinct} distinct")

        # 3) Load: merge the good rows; keep the current rejects for review
        if not spark.catalog.tableExists(silver_table):
            valid.write.format("delta").saveAsTable(silver_table)
        else:
            (DeltaTable.forName(spark, silver_table).alias("t")
                .merge(valid.alias("s"), f"t.{key} = s.{key}")
                # an older version must never overwrite a newer one (safe for reruns and replays)
                .whenMatchedUpdateAll(condition=f"s.{order_col} >= t.{order_col}")
                .whenNotMatchedInsertAll()
                .execute())

        # The rejects table shows what is failing right now, so it is rebuilt each run.
        rejected.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(rejects_table)

        rows_silver = spark.table(silver_table).count()
        log_run(status="success", rows_read=rows_bronze, rows_written=n_valid, **info)
        print(f"[{source_name}] bronze rows: {rows_bronze}  latest per key: {n_valid + n_rejected}  "
              f"valid: {n_valid}  rejected: {n_rejected}  silver rows now: {rows_silver}")
        if n_rejected:
            print(f"[{source_name}] review the rejects: SELECT reject_reason, COUNT(*) FROM {rejects_table} GROUP BY 1")

    except Exception as err:
        try:
            log_run(status="failed", error_message=str(err)[:2000], **info)
        except Exception as log_err:
            print(f"Could not write the failure row to meta.run_log: {log_err}")
        raise

In [ ]:
run(source_name)

## 6. Checks to run by hand after a load
```sql
-- one row per key?
SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS distinct_ids FROM silver.github_issues;

-- why were rows rejected?
SELECT reject_reason, COUNT(*) FROM silver.github_issues_rejects GROUP BY reject_reason;

-- bronze keeps every version, silver keeps the latest
SELECT (SELECT COUNT(*) FROM bronze.github_issues) AS bronze_rows,
       (SELECT COUNT(*) FROM silver.github_issues) AS silver_rows;
```

## Notes for production
- **Order of notebooks:** the pipeline runs the bronze notebook, then this one, per source.
- **Size:** this version re-reads all of bronze each run, which is fine for practice. For large tables, process only batches newer than the last silver load.
- **Gold** is built from silver (aggregates, joins, reporting tables), and is not part of this notebook.
- **Reject routing:** at the company, check where rejected rows should go and who reviews them.